In [3]:
%load_ext autoreload 
%autoreload 2

from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from datetime import datetime
from pyetc_ifs import ETC,WST,iredMUSE, MUSE, blueMUSE, get_seeing_fwhm, __version__
from scipy.optimize import brentq
from tqdm.notebook import tqdm
import json, pickle
import os
plt.rcParams.update({'font.size': 12})
file_dir = "/home/nbouche/Python_git/pyetc_iredmuse/images/"
print(f"pyetc_ifs version {__version__}")

pyetc_ifs version 0.1


In [4]:
airmasses=[1.0,1.2,1.5,2.0]
water=[1.0,3.5,10.0]


In [22]:
import json
import subprocess

# Define the configuration payload 
# For valid keys, refer to ESO's SkyCalc parameter lists
# https://www.eso.org/observing/etc/doc/skycalc/helpskycalccli.html
config = {
    "airmass": 1.0,
    "pwv_mode": "pwv",
    "pwv": 1.0,
    "incl_moon": "Y",
    "moon_alt": 45.0,
    "moon_target_sep":45.0,
    "moon_earth_dist":1.0,
    "moon_sun_sep":180.0,
    'incl_starlight':'Y',
    'incl_zodiacal':'Y',
    'ecl_lon':135.0,
    'ecl_lat':90.0,
    'incl_loweratm':'Y',
    'incl_upperatm':'Y',
    'incl_airglow':'Y',
    'incl_therm':'N',
    "wmin": 300.0,
    "wmax": 2000.0,
    "vacair": 'vac',
    "wgrid_mode": 'fixed_wavelength_step',
    "wdelta": 0.01,
    "wres": 1000,
    "lsf_type": 'none',
    "lsf_gauss_fwhm": 5.0,
    "lsf_boxcar_fwhm": 5.0,
}


In [23]:
config['moon_sun_sep']=180.0

In [24]:
def compute_sky(moon='dark'):
    if moon=='bright':
        config['moon_sun_sep']=180.0
    elif moon=='grey':
        config['moon_sun_sep']=90.0
    elif moon=='dark':
        config['moon_sun_sep']=0.0

    for X in airmasses:
        for pwv in water:
            config['airmass']=X
            config['pwv']=pwv
            output_fits="{}sky_{:.1f}_{:.1f}.fits".format(moon,X,pwv)
            skyinput="{}sky_{:.1f}_{:.1f}.json".format(moon,X,pwv)
            
            # Write the parameters to a local text or JSON configuration file
            with open(skyinput, "w") as f:
                json.dump(config, f, indent=4)
            
            # Call the skycalc_cli command using python's subprocess
            cmd = f"skycalc_cli -i {skyinput} -o {output_fits}"
            print(f"Running {cmd}")
                
            print(f"Querying ESO SkyCalc at airmass {config['airmass']}...")
            subprocess.run(cmd, shell=True, check=True)
            print(f"Success! Output generated in {output_fits}")
            #os.system("mv *sky ../data/sky/")
            

# bright sky

In [25]:
compute_sky('bright')

Running skycalc_cli -i brightsky_1.0_1.0.json -o brightsky_1.0_1.0.fits
Querying ESO SkyCalc at airmass 1.0...


/home/nbouche/miniconda3/lib/python3.13/site-packages/skycalc_cli/skycalc_cli.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in brightsky_1.0_1.0.fits
Running skycalc_cli -i brightsky_1.0_3.5.json -o brightsky_1.0_3.5.fits
Querying ESO SkyCalc at airmass 1.0...


/home/nbouche/miniconda3/lib/python3.13/site-packages/skycalc_cli/skycalc_cli.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in brightsky_1.0_3.5.fits
Running skycalc_cli -i brightsky_1.0_10.0.json -o brightsky_1.0_10.0.fits
Querying ESO SkyCalc at airmass 1.0...


...

Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in brightsky_2.0_1.0.fits
Running skycalc_cli -i brightsky_2.0_3.5.json -o brightsky_2.0_3.5.fits
Querying ESO SkyCalc at airmass 2.0...


/home/nbouche/miniconda3/lib/python3.13/site-packages/skycalc_cli/skycalc_cli.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in brightsky_2.0_3.5.fits
Running skycalc_cli -i brightsky_2.0_10.0.json -o brightsky_2.0_10.0.fits
Querying ESO SkyCalc at airmass 2.0...


/home/nbouche/miniconda3/lib/python3.13/site-packages/skycalc_cli/skycalc_cli.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in brightsky_2.0_10.0.fits


# grey

In [26]:
compute_sky('grey')

Running skycalc_cli -i greysky_1.0_1.0.json -o greysky_1.0_1.0.fits
Querying ESO SkyCalc at airmass 1.0...


/home/nbouche/miniconda3/lib/python3.13/site-packages/skycalc_cli/skycalc_cli.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in greysky_1.0_1.0.fits
Running skycalc_cli -i greysky_1.0_3.5.json -o greysky_1.0_3.5.fits
Querying ESO SkyCalc at airmass 1.0...


/home/nbouche/miniconda3/lib/python3.13/site-packages/skycalc_cli/skycalc_cli.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in greysky_1.0_3.5.fits
Running skycalc_cli -i greysky_1.0_10.0.json -o greysky_1.0_10.0.fits
Querying ESO SkyCalc at airmass 1.0...


...

Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in greysky_2.0_1.0.fits
Running skycalc_cli -i greysky_2.0_3.5.json -o greysky_2.0_3.5.fits
Querying ESO SkyCalc at airmass 2.0...


/home/nbouche/miniconda3/lib/python3.13/site-packages/skycalc_cli/skycalc_cli.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in greysky_2.0_3.5.fits
Running skycalc_cli -i greysky_2.0_10.0.json -o greysky_2.0_10.0.fits
Querying ESO SkyCalc at airmass 2.0...


/home/nbouche/miniconda3/lib/python3.13/site-packages/skycalc_cli/skycalc_cli.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in greysky_2.0_10.0.fits


# dark

In [27]:
compute_sky('dark')

Running skycalc_cli -i darksky_1.0_1.0.json -o darksky_1.0_1.0.fits
Querying ESO SkyCalc at airmass 1.0...


/home/nbouche/miniconda3/lib/python3.13/site-packages/skycalc_cli/skycalc_cli.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in darksky_1.0_1.0.fits
Running skycalc_cli -i darksky_1.0_3.5.json -o darksky_1.0_3.5.fits
Querying ESO SkyCalc at airmass 1.0...


/home/nbouche/miniconda3/lib/python3.13/site-packages/skycalc_cli/skycalc_cli.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in darksky_1.0_3.5.fits
Running skycalc_cli -i darksky_1.0_10.0.json -o darksky_1.0_10.0.fits
Querying ESO SkyCalc at airmass 1.0...


...

Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in darksky_2.0_1.0.fits
Running skycalc_cli -i darksky_2.0_3.5.json -o darksky_2.0_3.5.fits
Querying ESO SkyCalc at airmass 2.0...


/home/nbouche/miniconda3/lib/python3.13/site-packages/skycalc_cli/skycalc_cli.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in darksky_2.0_3.5.fits
Running skycalc_cli -i darksky_2.0_10.0.json -o darksky_2.0_10.0.fits
Querying ESO SkyCalc at airmass 2.0...


/home/nbouche/miniconda3/lib/python3.13/site-packages/skycalc_cli/skycalc_cli.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Note: skycalc_cli v.1.4 output wavelength unit is nm (it is μm in v.1.3)
Success! Output generated in darksky_2.0_10.0.fits


In [28]:
!ls grey* -lt

-rw-rw-r-- 1 nbouche nbouche 24503040 Jul 10 17:16 greysky_2.0_10.0.fits
-rw-rw-r-- 1 nbouche nbouche      605 Jul 10 17:16 greysky_2.0_10.0.json
-rw-rw-r-- 1 nbouche nbouche 24503040 Jul 10 17:16 greysky_2.0_3.5.fits
-rw-rw-r-- 1 nbouche nbouche      604 Jul 10 17:16 greysky_2.0_3.5.json
-rw-rw-r-- 1 nbouche nbouche 24503040 Jul 10 17:16 greysky_2.0_1.0.fits
-rw-rw-r-- 1 nbouche nbouche      604 Jul 10 17:16 greysky_2.0_1.0.json
-rw-rw-r-- 1 nbouche nbouche 24503040 Jul 10 17:16 greysky_1.5_10.0.fits
-rw-rw-r-- 1 nbouche nbouche      605 Jul 10 17:16 greysky_1.5_10.0.json
-rw-rw-r-- 1 nbouche nbouche 24503040 Jul 10 17:16 greysky_1.5_3.5.fits
-rw-rw-r-- 1 nbouche nbouche      604 Jul 10 17:16 greysky_1.5_3.5.json
-rw-rw-r-- 1 nbouche nbouche 24503040 Jul 10 17:16 greysky_1.5_1.0.fits
-rw-rw-r-- 1 nbouche nbouche      604 Jul 10 17:16 greysky_1.5_1.0.json
-rw-rw-r-- 1 nbouche nbouche 24503040 Jul 10 17:16 greysky_1.2_10.0.fits
-rw-rw-r-- 1 nbouche nbouche      605 Jul 10 17:15 greysky_

In [29]:
!pwd

/home/nbouche/Python_git/pyetc_iredmuse/pyetc_ifs/data/sky
